# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Azure AI Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

Before running this notebook, make sure you have:

1. **An Azure AI Foundry project** with a deployed chat model (e.g. `gpt-4o-mini`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **Set the required environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — your Azure AI Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.

The cell below installs the Python packages you need.

In [1]:
%pip install agent-framework azure-ai-projects azure-identity -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

c:\Users\elocusteanu\Documents\FY26\GAIACADEMY\AI-For-Beginners\.venv312\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Users\elocusteanu\Documents\FY26\GAIACADEMY\AI-For-Beginners\.venv312\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [3]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [4]:
@tool(approval_mode="never_require")
def get_destinations_2() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Amsterdam",
        "Athens",
        "Milan",
        ]

In [4]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

For a warm beach destination, I recommend considering Barcelona, Sydney, Rio de Janeiro, or Bali. These locations are known for their beautiful beaches and warm climates. Would you like more information on any of these destinations?


In [5]:
agent_2 = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations_2 tool "
        "to see available destinations."
    ),
    tools=[get_destinations_2],
)

response = await agent_2.run(
    "I'm looking for a place with cheap good food. What do you recommend? Pick one destination only."
)
print(response)

Among Amsterdam, Athens, and Milan, I recommend Athens for cheap good food. Athens is known for its delicious and affordable Greek cuisine, offering a variety of tasty options like souvlaki, gyros, and fresh seafood at reasonable prices. Would you like more information about Athens or help planning your trip there?


## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [7]:
async for chunk in agent_2.run("Tell me about Amsterdam as a travel destination", stream=True):
    print(chunk, end="", flush=True)

Amsterdam is a vibrant and charming city known for its picturesque canals, rich history, and cultural attractions. It's a popular travel destination for people who enjoy art, history, and unique urban experiences. Some highlights of Amsterdam include:

1. Canals: The city's canal system is iconic and offers lovely boat tours or leisurely walks along the water.
2. Museums: Amsterdam is home to world-renowned museums such as the Rijksmuseum, Van Gogh Museum, and Anne Frank House.
3. Architecture: The historic buildings and narrow houses with gabled facades give the city its unique character.
4. Bicycling: Amsterdam is a bike-friendly city with extensive cycling paths, making it easy and fun to explore.
5. Cafés and Markets: The city has charming cafés, street markets, and diverse culinary options.
6. Nightlife: Offers a range of lively bars, clubs, and entertainment venues.

Amsterdam is ideal for travelers who appreciate culture, history, and a relaxed yet energetic urban atmosphere. Wo

In [6]:
response = await agent_2.run("Tell me about Amsterdam as a travel destination")
print(response)

Amsterdam is a vibrant and charming city known for its picturesque canals, historic architecture, and rich cultural heritage. It's famous for its art museums such as the Rijksmuseum and the Van Gogh Museum, showcasing masterpieces from renowned artists. Visitors can explore the Anne Frank House to learn about history and human resilience.

The city offers a lively atmosphere with numerous cafes, restaurants, and shops. Biking is a popular way to get around and experience local life. Amsterdam is also known for its unique neighborhoods like Jordaan, with trendy boutiques and cozy cafes.

Whether you're interested in history, art, or just enjoying a relaxed urban vibe, Amsterdam offers a diverse range of activities and attractions for travelers. If you want, I can provide more detailed recommendations or help plan your visit.


## Summary

In this lesson you learned how to:

- **Create a provider** that connects to Azure AI Foundry Agent Service via `AzureAIProjectAgentProvider`.
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Run the agent** with a user message and print its response.
- **Stream responses** for real-time output.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.